In [35]:
import pandas as pd
import plotly.express as px
import json
import numpy as np

from dash import Dash, html, dcc, Input, Output, dash_table, no_update

In [36]:
dataset = pd.read_csv("world_economics.csv")
dataset

,name,currency,capital,languages,latitude,longitude,area,region,subregion,borders,GDP,GDP Growth,Interest Rate,Inflation Rate,Jobless Rate,Gov. Budget,Debt/GDP,Current Account,Population
0,Tunisia,Tunisian dinar,Tunis,Arabic,34.000000,9.000000,163610.0,Africa,Northern Africa,"['DZA', 'LBY']",53,1.80,7.50,5.30,15.70,-6.6,79.8,-2.7,11.89
1,Vietnam,Vietnamese đồng,Hanoi,Vietnamese,16.166667,107.833333,331212.0,Asia,South-Eastern Asia,"['KHM', 'CHN', 'LAO']",476,6.88,4.50,3.19,2.24,-4.0,32.9,6.1,101.30
2,Ecuador,United States dollar,Quito,Spanish,-2.000000,-77.500000,276841.0,Americas,South America,"['COL', 'PER']",125,3.46,8.50,0.72,4.50,-1.9,50.6,2.8,17.97
3,Puerto Rico,United States dollar,San Juan,English,18.250000,-66.500000,8870.0,Americas,Caribbean,NaN,126,NaN,NaN,1.20,5.70,NaN,NaN,NaN,3.20
4,United States,United States dollar,"Washington, D.C.",English,38.000000,-97.000000,9372610.0,Americas,North America,"['CAN', 'MEX']",29185,3.00,4.50,2.70,4.20,-6.4,124.3,-3.9,341.15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,New Caledonia,CFP franc,Nouméa,French,-21.500000,165.500000,18575.0,Oceania,Melanesia,NaN,10,NaN,2.00,0.80,11.20,NaN,NaN,NaN,0.29
169,Kosovo,Euro,Pristina,Albanian,42.666667,21.166667,10908.0,Europe,Southeast Europe,"['ALB', 'MKD', 'MNE', 'SRB']",11,NaN,NaN,4.30,10.80,0.2,16.9,-9.0,1.60
170,Malaysia,Malaysian ringgit,Kuala Lumpur,English,2.500000,112.500000,330803.0,Asia,South-Eastern Asia,"['BRN', 'IDN', 'THA']",422,2.10,2.75,1.20,3.00,-4.1,70.4,1.7,34.10
171,Fiji,Fijian dollar,Suva,English,-17.713400,178.065000,18272.0,Oceania,Melanesia,NaN,6,NaN,0.25,-0.40,4.30,-3.4,78.3,-8.7,0.93


In [37]:
dataset.isna().sum()

name                0
currency            0
capital             1
languages           0
latitude            0
longitude           0
area                0
region              0
subregion           0
borders            24
GDP                 0
GDP Growth         72
Interest Rate      11
Inflation Rate      1
Jobless Rate        2
Gov. Budget         7
Debt/GDP            8
Current Account     6
Population          0
dtype: int64

In [38]:
dataset.drop(columns=['borders'], inplace=True)
dataset.columns

Index(['name', 'currency', 'capital', 'languages', 'latitude', 'longitude',
       'area', 'region', 'subregion', 'GDP', 'GDP Growth', 'Interest Rate',
       'Inflation Rate', 'Jobless Rate', 'Gov. Budget', 'Debt/GDP',
       'Current Account', 'Population'],
      dtype='object')

In [39]:
dataset['GDP_log'] = np.log10(dataset['GDP'])

In [40]:
world_map = json.load(open('world.geojson'))

In [41]:
countries = []

for feature in world_map["features"]:
    countries.append(feature["properties"]["NAME"])

countries = list(set(countries))
countries.sort()


In [42]:
country_df = pd.DataFrame(countries, columns =['Country'])

In [43]:
metrics = {
    "GDP": "GDP",
    "Interest Rate": "Interest Rate",
    "Inflation Rate": "Inflation Rate",
    "Jobless Rate": "Jobless Rate",
    "Debt/GDP": "Debt/GDP",
    "Population": "Population"
}

In [83]:
app = Dash(__name__)

app.layout = html.Div(
    style={
        "backgroundColor": "white",
        "minHeight": "100vh",
        "color": "black",
        "padding": "15px"
    },
    children=[
        html.H1('Data', style={"margin": "0 0 10px 0"}),

        # ---------------- Inline dropdowns ----------------
        html.Div(
            style={"display": "flex", "gap": "15px", "marginBottom": "20px"},
            children=[
                dcc.Dropdown(
                    id="metric",
                    options=[{"label": k, "value": v} for k, v in metrics.items()],
                    value="GDP",
                    clearable=False,
                    style={"width": "250px"}
                ),
                dcc.Dropdown(
                    id="region",
                    options=[
                        {"label": "Europe", "value": "Europe"},
                        {"label": "Asia", "value": "Asia"},
                        {"label": "Africa", "value": "Africa"},
                        {"label": "Americas", "value": "Americas"},
                        {"label": "Oceania", "value": "Oceania"},
                    ],
                    value=[],  # default: no region filter
                    multi=True,
                    clearable=True,
                    style={"width": "250px"}
                ),
                html.Button("Reset", id="reset", n_clicks=0),
            ]
        ),

        html.Div(
            style={"display": "flex", "gap": "20px", "alignItems": "stretch"},
            children=[
                dcc.Graph(id="map", style={"height": "60vh", "flex": "2"}),
                dcc.Graph(id="spider", style={"height": "60vh", "flex": "1"}),
            ]
        ),

        dcc.Graph(id="bar", style={"height": "40vh"})
    ]
)

def get_map(data, metric):
    # ---------------- FIX 1: avoid log10(0) / -inf ----------------
    data = data[data[metric] > 0].copy()
    data["value"] = data[metric]
    data["value_log"] = np.log10(data["value"])

    fig = px.choropleth(
        data,
        geojson=world_map,
        featureidkey="properties.NAME",
        locations="name",
        color="value_log",
        hover_data={metric: ":,.0f", "value_log": False},
        hover_name="name",
        custom_data=["name"],
        color_continuous_scale="Viridis",
        projection="orthographic",
        title=f"{metric} by Country"
    )

    fig.update_geos(
        visible=False,
        projection_type="orthographic",
        showcountries=True,
        domain=dict(x=[0.05, 0.95], y=[0.05, 0.95])
    )

    # Colorbar with log → original values
    vmin, vmax = data["value_log"].min(), data["value_log"].max()
    ticks = np.linspace(vmin, vmax, 6)
    fig.update_coloraxes(
        colorbar_tickvals=ticks,
        colorbar_ticktext=[f"{10**v:,.0f}" for v in ticks],
        colorbar_title=metric,
        colorbar=dict(thickness=20, len=0.3, x=0.9, y=0.75)
    )

    fig.update_layout(
        paper_bgcolor="white",
        plot_bgcolor="white",
        geo_bgcolor="white",
        margin=dict(t=50, b=0, l=0, r=0),
        font_color="black"
    )

    return fig


def get_barchart(data, metric, top_n=None):
    data = data[data[metric] > 0].copy()

    # Optional: show only top N countries
    if top_n:
        data = data.sort_values(metric, ascending=False).head(top_n)
    else:
        data = data.sort_values(metric, ascending=True)  # ascending for horizontal bar chart

    fig = px.bar(
        data,
        x=metric,
        y="name",
        orientation='h',  # horizontal
        text=metric,
        labels={"name": "Country", metric: metric},
        title=f"{metric} by Country"
    )

    fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside')

    # Dynamically adjust height based on number of countries
    height_per_country = 30  # pixels per country
    chart_height = max(400, len(data) * height_per_country)

    fig.update_layout(
        height=chart_height,
        margin=dict(t=50, b=50, l=150, r=50),  # leave space for country names
        yaxis=dict(title="Country"),
        xaxis=dict(title=metric)
    )

    return fig


def get_spider(data, country_name):
    # List of metrics for the radar chart
    radar_metrics = [
        "GDP",
        "Inflation Rate",
        "Unemployment Rate",
        "Gov Budget",
        "Debt"
    ]

    # Use real column names from the dropdown metrics dictionary
    radar_metrics = list(metrics.values())

    # If there are not enough metrics, return empty figure
    if len(radar_metrics) < 3:
        return px.line_polar(title="Not enough metrics for spider chart")

    # Country row
    row = data[data["name"] == country_name]
    if row.empty:
        return px.line_polar(title="Country not found")

    row = row.iloc[0]

    # Extract numeric values
    values = []
    for m in radar_metrics:
        v = pd.to_numeric(row[m], errors="coerce")
        values.append(v)

    if all(pd.isna(v) for v in values):
        return px.line_polar(title="No numeric data for this country")

    # Min-max normalization
    norm_values = []
    for m, v in zip(radar_metrics, values):
        col = pd.to_numeric(data[m], errors="coerce")
        mn = col.min(skipna=True)
        mx = col.max(skipna=True)

        if pd.isna(v) or pd.isna(mn) or pd.isna(mx) or mx == mn:
            norm_values.append(0)
        else:
            norm_values.append((v - mn) / (mx - mn))

    spider_df = pd.DataFrame({
        "metric": radar_metrics,
        "value": norm_values
    })

    fig = px.line_polar(
        spider_df,
        r="value",
        theta="metric",
        line_close=True,
        title=f"Spider Chart — {country_name} (normalized)"
    )

    fig.update_traces(fill="toself")
    fig.update_layout(
        paper_bgcolor="white",
        plot_bgcolor="white",
        font_color="black",
        margin=dict(t=60, b=20, l=40, r=40)
    )

    return fig


@app.callback(
    Output("map", "figure"),
    Output("bar", "figure"),
    Input("metric", "value"),
    Input("region", "value")
)
def update_figures(metric, regions):
    data = dataset.copy()
    if regions and len(regions) > 0:
        data = data[data["region"].isin(regions)].copy()
    map_fig = get_map(data, metric)
    bar_fig = get_barchart(data, metric)
    return map_fig, bar_fig


@app.callback(
    Output("metric", "value"),
    Output("region", "value"),
    Input("reset", "n_clicks"),
    prevent_initial_call=True
)
def reset_all(n):
    return "GDP", []


@app.callback(
    Output("spider", "figure"),
    Input("map", "clickData"),
    Input("metric", "value"),
    Input("region", "value"),
)
def update_spider(clickData, metric, regions):
    if not clickData:
        return px.line_polar(title="Click a country on the map to see the spider chart")

    point = clickData["points"][0]
    country = (
        point.get("location")
        or point.get("hovertext")
        or (point.get("customdata")[0] if point.get("customdata") else None)
    )

    if not country:
        return px.line_polar(title="Could not read country from click")

    data = dataset.copy()
    if regions and len(regions) > 0:
        data = data[data["region"].isin(regions)].copy()

    return get_spider(data, str(country).strip())


app.run(jupyter_mode="external", debug=True)


Dash app running on http://127.0.0.1:8050/


In [84]:
dataset["GDP_log"].min()

np.float64(0.3010299956639812)

In [85]:
dataset["GDP_log"].max()

np.float64(4.465159697646179)

In [86]:
dataset['region'].unique()

array(['Africa', 'Asia', 'Americas', 'Europe', 'Oceania'], dtype=object)